<a href="https://colab.research.google.com/github/Riley-McDonald/Airline-Case-Study-Problems/blob/main/FlightCase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Case 1 — Flight Delay Cascade Detection
It's 7:42 AM on a Tuesday. JFK is showing a cluster of delays. Your director says: "Find me the aircraft that are going to take this network-wide before it turns into cancellations."

In [70]:
#import pandas and numpy, read in excel file and assign datetime

import pandas as pd
import numpy as np

df = pd.read_csv("case1_flights.csv", parse_dates=['scheduled_dep','actual_dep'])
df.head()

,flight_num,origin,dest,scheduled_dep,actual_dep,tail_num,status
0,B6 415,JFK,LAX,2024-01-16 06:00:00,2024-01-16 06:47:00,N729JB,delayed
1,B6 832,LAX,SFO,2024-01-16 09:30:00,2024-01-16 10:22:00,N729JB,delayed
2,B6 201,SFO,BOS,2024-01-16 13:00:00,2024-01-16 13:58:00,N729JB,delayed
3,B6 118,JFK,FLL,2024-01-16 07:00:00,2024-01-16 07:12:00,N304JB,delayed
4,B6 445,FLL,MCO,2024-01-16 10:00:00,2024-01-16 10:08:00,N304JB,on-time


In [75]:
#check for nan values

df.isna().sum()

,0
flight_num,0
origin,0
dest,0
scheduled_dep,0
actual_dep,0
tail_num,0
status,0
delay_minutes,0


In [76]:
#check for duplicates

len(df['flight_num'].unique())

21

In [71]:
#calculate delay based on actual dep and scheduled dep

df['delay']=df['actual_dep']-df['scheduled_dep']
df.head()

,flight_num,origin,dest,scheduled_dep,actual_dep,tail_num,status,delay
0,B6 415,JFK,LAX,2024-01-16 06:00:00,2024-01-16 06:47:00,N729JB,delayed,0 days 00:47:00
1,B6 832,LAX,SFO,2024-01-16 09:30:00,2024-01-16 10:22:00,N729JB,delayed,0 days 00:52:00
2,B6 201,SFO,BOS,2024-01-16 13:00:00,2024-01-16 13:58:00,N729JB,delayed,0 days 00:58:00
3,B6 118,JFK,FLL,2024-01-16 07:00:00,2024-01-16 07:12:00,N304JB,delayed,0 days 00:12:00
4,B6 445,FLL,MCO,2024-01-16 10:00:00,2024-01-16 10:08:00,N304JB,on-time,0 days 00:08:00


In [72]:
#calculate delay minutes using dt.total_seconds()/60

df['delay_minutes']=(df['actual_dep']-df['scheduled_dep']).dt.total_seconds()/60
df.head()

,flight_num,origin,dest,scheduled_dep,actual_dep,tail_num,status,delay,delay_minutes
0,B6 415,JFK,LAX,2024-01-16 06:00:00,2024-01-16 06:47:00,N729JB,delayed,0 days 00:47:00,47.0
1,B6 832,LAX,SFO,2024-01-16 09:30:00,2024-01-16 10:22:00,N729JB,delayed,0 days 00:52:00,52.0
2,B6 201,SFO,BOS,2024-01-16 13:00:00,2024-01-16 13:58:00,N729JB,delayed,0 days 00:58:00,58.0
3,B6 118,JFK,FLL,2024-01-16 07:00:00,2024-01-16 07:12:00,N304JB,delayed,0 days 00:12:00,12.0
4,B6 445,FLL,MCO,2024-01-16 10:00:00,2024-01-16 10:08:00,N304JB,on-time,0 days 00:08:00,8.0


In [73]:
#drop the delay column, no need for duplicate info

df = df.drop(columns=['delay'])
df.head()

,flight_num,origin,dest,scheduled_dep,actual_dep,tail_num,status,delay_minutes
0,B6 415,JFK,LAX,2024-01-16 06:00:00,2024-01-16 06:47:00,N729JB,delayed,47.0
1,B6 832,LAX,SFO,2024-01-16 09:30:00,2024-01-16 10:22:00,N729JB,delayed,52.0
2,B6 201,SFO,BOS,2024-01-16 13:00:00,2024-01-16 13:58:00,N729JB,delayed,58.0
3,B6 118,JFK,FLL,2024-01-16 07:00:00,2024-01-16 07:12:00,N304JB,delayed,12.0
4,B6 445,FLL,MCO,2024-01-16 10:00:00,2024-01-16 10:08:00,N304JB,on-time,8.0


In [80]:
#add a delayed flag column for flights delayed over 15 minutes

df['is_delayed']=df['delay_minutes']>15
df.head()

,flight_num,origin,dest,scheduled_dep,actual_dep,tail_num,status,delay_minutes,is_delayed
19,B6 777,JFK,MCO,2024-01-16 06:45:00,2024-01-16 06:50:00,N100JB,on-time,5.0,False
20,B6 778,MCO,FLL,2024-01-16 10:15:00,2024-01-16 10:20:00,N100JB,on-time,5.0,False
9,B6 900,JFK,LGB,2024-01-16 08:00:00,2024-01-16 08:05:00,N198JB,on-time,5.0,False
10,B6 921,LGB,JFK,2024-01-16 12:00:00,2024-01-16 12:03:00,N198JB,on-time,3.0,False
3,B6 118,JFK,FLL,2024-01-16 07:00:00,2024-01-16 07:12:00,N304JB,delayed,12.0,False


In [81]:
#cacluate mean delay per leg to visualize which tails are problems

mean_per_leg = df.groupby('tail_num')['delay_minutes'].mean()
print(mean_per_leg)

tail_num
N100JB      5.000000
N198JB      4.000000
N304JB      6.000000
N445JB      7.666667
N563JB    115.666667
N677JB    131.500000
N729JB     52.333333
N812JB    100.666667
Name: delay_minutes, dtype: float64


In [82]:
#sort the data frame by tail_num then by scheduled_dep

df = df.sort_values(['tail_num','scheduled_dep'])
df.head()

,flight_num,origin,dest,scheduled_dep,actual_dep,tail_num,status,delay_minutes,is_delayed
19,B6 777,JFK,MCO,2024-01-16 06:45:00,2024-01-16 06:50:00,N100JB,on-time,5.0,False
20,B6 778,MCO,FLL,2024-01-16 10:15:00,2024-01-16 10:20:00,N100JB,on-time,5.0,False
9,B6 900,JFK,LGB,2024-01-16 08:00:00,2024-01-16 08:05:00,N198JB,on-time,5.0,False
10,B6 921,LGB,JFK,2024-01-16 12:00:00,2024-01-16 12:03:00,N198JB,on-time,3.0,False
3,B6 118,JFK,FLL,2024-01-16 07:00:00,2024-01-16 07:12:00,N304JB,delayed,12.0,False


In [83]:
#add in prev_delayed column grouped by tail_num and delay flag
#flag tells us if the prev flight from the same tail was delayed

df['prev_delayed']=df.groupby('tail_num')['is_delayed'].shift(1)
df.head()

,flight_num,origin,dest,scheduled_dep,actual_dep,tail_num,status,delay_minutes,is_delayed,prev_delayed
19,B6 777,JFK,MCO,2024-01-16 06:45:00,2024-01-16 06:50:00,N100JB,on-time,5.0,False,NaN
20,B6 778,MCO,FLL,2024-01-16 10:15:00,2024-01-16 10:20:00,N100JB,on-time,5.0,False,False
9,B6 900,JFK,LGB,2024-01-16 08:00:00,2024-01-16 08:05:00,N198JB,on-time,5.0,False,NaN
10,B6 921,LGB,JFK,2024-01-16 12:00:00,2024-01-16 12:03:00,N198JB,on-time,3.0,False,False
3,B6 118,JFK,FLL,2024-01-16 07:00:00,2024-01-16 07:12:00,N304JB,delayed,12.0,False,NaN


In [89]:
#make df called at_risk including flights where the
#current tails flight and its previous flight are both delayed to find
#flights in delay prop

at_risk = df[(df['is_delayed']==True)&(df['prev_delayed']==True)]
at_risk.head()

,flight_num,origin,dest,scheduled_dep,actual_dep,tail_num,status,delay_minutes,is_delayed,prev_delayed
7,B6 711,JFK,DCA,2024-01-16 10:00:00,2024-01-16 11:52:00,N563JB,delayed,112.0,True,True
8,B6 555,DCA,BOS,2024-01-16 14:00:00,2024-01-16 16:10:00,N563JB,delayed,130.0,True,True
18,B6 611,SFO,LGB,2024-01-16 12:00:00,2024-01-16 14:18:00,N677JB,delayed,138.0,True,True
1,B6 832,LAX,SFO,2024-01-16 09:30:00,2024-01-16 10:22:00,N729JB,delayed,52.0,True,True
2,B6 201,SFO,BOS,2024-01-16 13:00:00,2024-01-16 13:58:00,N729JB,delayed,58.0,True,True


In [85]:
# use the at_risk df to find unique tail numbers included
# essentailly the tail numbers going thru delay prop

at_risk_tails = at_risk['tail_num'].unique()
print(at_risk_tails)

['N563JB' 'N677JB' 'N729JB' 'N812JB']


In [90]:
#make new df named risk_flights to find all flights the at risk
#tails have flown

risk_flights = df[df['tail_num'].isin(at_risk_tails)]
risk_flights.head()

,flight_num,origin,dest,scheduled_dep,actual_dep,tail_num,status,delay_minutes,is_delayed,prev_delayed
6,B6 302,BOS,JFK,2024-01-16 06:30:00,2024-01-16 08:15:00,N563JB,delayed,105.0,True,NaN
7,B6 711,JFK,DCA,2024-01-16 10:00:00,2024-01-16 11:52:00,N563JB,delayed,112.0,True,True
8,B6 555,DCA,BOS,2024-01-16 14:00:00,2024-01-16 16:10:00,N563JB,delayed,130.0,True,True
17,B6 610,JFK,SFO,2024-01-16 07:00:00,2024-01-16 09:05:00,N677JB,delayed,125.0,True,NaN
18,B6 611,SFO,LGB,2024-01-16 12:00:00,2024-01-16 14:18:00,N677JB,delayed,138.0,True,True


In [91]:
#print summary detailing number of aircraft in delay prop
#list of aircraft flight num and route of delay prop flights

print(f"Aircarft at propagation risk: {len(at_risk_tails)}")
print(risk_flights[['tail_num','flight_num','origin','dest','delay_minutes']])

Aircarft at propagation risk: 4
   tail_num flight_num origin dest  delay_minutes
6    N563JB     B6 302    BOS  JFK          105.0
7    N563JB     B6 711    JFK  DCA          112.0
8    N563JB     B6 555    DCA  BOS          130.0
17   N677JB     B6 610    JFK  SFO          125.0
18   N677JB     B6 611    SFO  LGB          138.0
0    N729JB     B6 415    JFK  LAX           47.0
1    N729JB     B6 832    LAX  SFO           52.0
2    N729JB     B6 201    SFO  BOS           58.0
11   N812JB     B6 340    JFK  MCO           89.0
12   N812JB     B6 341    MCO  JFK           91.0
13   N812JB     B6 342    JFK  BOS          122.0
